# PyTorch CUDA Check

- see: [pyproject.toml#optional-dependencies](https://gitlab.com/ub-dems/ds-labs/dve-sample-r/-/blob/main/pyproject.toml#L156)


## GPU Autodetection


In [11]:
%%bash

(type nvidia-smi && nvidia-smi -L) &> /dev/null \
    && X_HAS_GPU=1 || X_HAS_GPU=0;

case "$X_HAS_GPU" in
     1) export X_UV_EXTRA='gpu' ;;
     *) export X_UV_EXTRA='cpu' ;;
esac
export X_HAS_GPU

echo "X_HAS_GPU=$X_HAS_GPU"
echo "X_UV_EXTRA=$X_UV_EXTRA"

X_HAS_GPU=1
X_UV_EXTRA=gpu


## NVIDIA CUDA Linkage

```bash
uv lock
uv sync --extra"$X_UV_EXTRA" --all-groups
```

In [12]:
! uv pip list | grep -e keras -e torch -e tensorflow -e ^nv

Using Python 3.14.8 environment at: /root/work/vb/dve-sample-r/.venv.cdk
keras                                    3.15.1
nvidia-cublas                            13.1.0.3
nvidia-cuda-cupti                        13.0.85
nvidia-cuda-nvrtc                        13.0.88
nvidia-cuda-runtime                      13.0.96
nvidia-cudnn-cu13                        9.15.1.9
nvidia-cufft                             12.0.0.61
nvidia-cufile                            1.15.1.6
nvidia-curand                            10.4.0.35
nvidia-cusolver                          12.0.4.66
nvidia-cusparse                          12.6.3.3
nvidia-cusparselt-cu13                   0.8.0
nvidia-ml-py                             13.615.71
nvidia-nccl-cu13                         2.28.9
nvidia-nvjitlink                         13.0.88
nvidia-nvshmem-cu13                      3.4.5
nvidia-nvtx                              13.0.85
torch                                    2.10.0+cu130
torchvision                       

## PyTorch Test

In [10]:
from __future__ import annotations
import argparse
from typing import Any

In [2]:
import torch
from torch import nn
import torch.nn.functional as F
from torch import optim

from torchvision import datasets, transforms
from torch.optim.lr_scheduler import StepLR

In [3]:
print(f"cuda.is_available={torch.cuda.is_available()}")

cuda.is_available=True


In [4]:
torch.cuda.init()

In [5]:
device = torch.cuda.current_device()
print(f"cuda.get_device_name={torch.cuda.get_device_name()}")
print(f"cuda.get_device_capability={torch.cuda.get_device_capability()}")
print(f"cuda.get_device_properties={torch.cuda.get_device_properties(device)}")

cuda.get_device_name=Tesla T4
cuda.get_device_capability=(7, 5)
cuda.get_device_properties=_CudaDeviceProperties(name='Tesla T4', major=7, minor=5, total_memory=15945MB, multi_processor_count=40, uuid=b66900c1-312c-96aa-bf64-0764873d0131, pci_bus_id=0, pci_device_id=0, pci_domain_id=1, L2_cache_size=4MB)


In [6]:
X_tensor = torch.FloatTensor([0.0, 1.0, 2.0])
Y_tensor = X_tensor.cuda()
print(f"X_tensor.is_cuda={X_tensor.is_cuda}")
print(f"Y_tensor.is_cuda={Y_tensor.is_cuda}")

X_tensor.is_cuda=False
Y_tensor.is_cuda=True


In [7]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x)
        output = F.log_softmax(x, dim=1)
        return output


# ruff: noqa: PLW2901
def train(args, model, device, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % args.log_interval == 0:
            print(
                "Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}".format(
                    epoch,
                    batch_idx * len(data),
                    len(train_loader.dataset),
                    100.0 * batch_idx / len(train_loader),
                    loss.item(),
                )
            )
            if args.dry_run:
                break


# ruff: noqa: PLW2901
def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction="sum").item()  # sum up batch loss
            pred = output.argmax(dim=1, keepdim=True)  # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)

    print(
        "\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n".format(
            test_loss, correct, len(test_loader.dataset), 100.0 * correct / len(test_loader.dataset)
        )
    )


def main(argv: list[str]):
    # Training settings
    parser = argparse.ArgumentParser(description="PyTorch MNIST Example")
    parser.add_argument(
        "--batch-size",
        type=int,
        default=64,
        metavar="N",
        help="input batch size for training (default: 64)",
    )
    parser.add_argument(
        "--test-batch-size",
        type=int,
        default=1000,
        metavar="N",
        help="input batch size for testing (default: 1000)",
    )
    parser.add_argument(
        "--epochs",
        type=int,
        default=14,
        metavar="N",
        help="number of epochs to train (default: 14)",
    )
    parser.add_argument("--lr", type=float, default=1.0, metavar="LR", help="learning rate (default: 1.0)")
    parser.add_argument(
        "--gamma",
        type=float,
        default=0.7,
        metavar="M",
        help="Learning rate step gamma (default: 0.7)",
    )
    parser.add_argument("--no-cuda", action="store_true", default=False, help="disables CUDA training")
    parser.add_argument("--no-mps", action="store_true", default=False, help="disables macOS GPU training")
    parser.add_argument("--dry-run", action="store_true", default=False, help="quickly check a single pass")
    parser.add_argument("--seed", type=int, default=1, metavar="S", help="random seed (default: 1)")
    parser.add_argument(
        "--log-interval",
        type=int,
        default=10,
        metavar="N",
        help="how many batches to wait before logging training status",
    )
    parser.add_argument("--save-model", action="store_true", default=False, help="For Saving the current Model")
    args = parser.parse_args(argv)
    use_cuda = not args.no_cuda and torch.cuda.is_available()
    use_mps = not args.no_mps and torch.backends.mps.is_available()

    torch.manual_seed(args.seed)

    if use_cuda:
        device = torch.device("cuda")
    elif use_mps:
        device = torch.device("mps")
    else:
        device = torch.device("cpu")

    train_kwargs: dict[str, Any] = {"batch_size": args.batch_size}
    test_kwargs: dict[str, Any] = {"batch_size": args.test_batch_size}

    if use_cuda:
        cuda_kwargs: dict[str, Any] = {"num_workers": 1, "pin_memory": True, "shuffle": True}
        train_kwargs.update(cuda_kwargs)
        test_kwargs.update(cuda_kwargs)

    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
    dataset1 = datasets.MNIST("../data", train=True, download=True, transform=transform)
    dataset2 = datasets.MNIST("../data", train=False, transform=transform)

    # Calculate values dynamically based on use_cuda
    workers = 1 if use_cuda else 0
    pin = True if use_cuda else False

    train_loader = torch.utils.data.DataLoader(
        dataset1, batch_size=args.batch_size, num_workers=workers, pin_memory=pin, shuffle=True
    )

    test_loader = torch.utils.data.DataLoader(
        dataset2, batch_size=args.test_batch_size, num_workers=workers, pin_memory=pin, shuffle=False
    )

    model = Net().to(device)
    optimizer = optim.Adadelta(model.parameters(), lr=args.lr)

    scheduler = StepLR(optimizer, step_size=1, gamma=args.gamma)
    for epoch in range(1, args.epochs + 1):
        train(args, model, device, train_loader, optimizer, epoch)
        test(model, device, test_loader)
        scheduler.step()

    if args.save_model:
        torch.save(model.state_dict(), "mnist_cnn.pt")

In [8]:
main(["--epochs=3"])

100%|██████████| 9.91M/9.91M [00:00<00:00, 15.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 353kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.32MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 31.6MB/s]


Train Epoch: 1 [0/60000 (0%)]	Loss: 2.282551
Train Epoch: 1 [640/60000 (1%)]	Loss: 1.385042
Train Epoch: 1 [1280/60000 (2%)]	Loss: 0.907695
Train Epoch: 1 [1920/60000 (3%)]	Loss: 0.626155
Train Epoch: 1 [2560/60000 (4%)]	Loss: 0.422388
Train Epoch: 1 [3200/60000 (5%)]	Loss: 0.461396
Train Epoch: 1 [3840/60000 (6%)]	Loss: 0.271166
Train Epoch: 1 [4480/60000 (7%)]	Loss: 0.725000
Train Epoch: 1 [5120/60000 (9%)]	Loss: 0.240226
Train Epoch: 1 [5760/60000 (10%)]	Loss: 0.368788
Train Epoch: 1 [6400/60000 (11%)]	Loss: 0.297282
Train Epoch: 1 [7040/60000 (12%)]	Loss: 0.199871
Train Epoch: 1 [7680/60000 (13%)]	Loss: 0.290360
Train Epoch: 1 [8320/60000 (14%)]	Loss: 0.171102
Train Epoch: 1 [8960/60000 (15%)]	Loss: 0.306520
Train Epoch: 1 [9600/60000 (16%)]	Loss: 0.194508
Train Epoch: 1 [10240/60000 (17%)]	Loss: 0.260597
Train Epoch: 1 [10880/60000 (18%)]	Loss: 0.295215
Train Epoch: 1 [11520/60000 (19%)]	Loss: 0.223825
Train Epoch: 1 [12160/60000 (20%)]	Loss: 0.146618
Train Epoch: 1 [12800/60000 (

In [9]:
device = torch.cuda.current_device()
print(f"cuda.utilization={torch.cuda.utilization(device)}")

cuda.utilization=0
